In [483]:
!pip install pyserial

In [484]:
import serial, time
!pip install pyserial

In [485]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [486]:
print(serial)

<module 'serial' from 'C:\\Users\\boome\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [487]:
print(serial.__file__)

C:\Users\boome\anaconda3\Lib\site-packages\serial\__init__.py


In [488]:
print(serial.__version__)

3.5


In [489]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [490]:
baudrate = 115200

In [491]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [492]:
#ser.close()

In [493]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [494]:
ser.in_waiting

0

In [495]:
#ser.close()

In [496]:
#read_all(ser)

In [497]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [498]:
read_all(ser)

''

In [499]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [500]:
read_one_line(ser)

'dual servo control over serial'

In [501]:
read_all(ser)

''

In [502]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [503]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

## Example

In [504]:
#byte1 = 7
#WriteByte(ser,byte1)#<--
#time.sleep(0.1)
#byte2 = 156
#WriteByte(ser,byte2)#<--
#time.sleep(0.1)
#next_line = read_one_line(ser)
#extra = read_all(ser)
#print('next_line: %s' % next_line)
#print('extra: %s' % extra)

# Break an integer into two bytes

In [505]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

In [506]:
# inputs from user
xll = 5 # x origin
yll = 5 # y origin
w = 10   # width
h = 10   # height
N = 10  # number of steps per side

In [507]:
#define step size
dx = w/N
dy = h/N

#generate bottom coordinants
x_bottom = np.linspace(xll, xll+w-dx, N)  
y_bottom = np.full(N, yll)

#generate right coordinants
x_right = np.full(N, xll + w)  
y_right = np.linspace(yll, yll+h-dy, N)

#generate top coordinants
x_top = np.linspace(xll+w, xll+dx, N)  
y_top = np.full(N,yll+h)

#generate left coordinants
x_left = np.full(N, xll)  
y_left = np.linspace(yll+h, yll, N)

#combine bottom, right, top, left into one array
x_path = np.concatenate((x_bottom, x_right, x_top, x_left), axis=0) 
y_path = np.concatenate((y_bottom, y_right, y_top, y_left), axis=0) 

#combine x and y into one array
tip_path = np.column_stack((x_path, y_path))
tip_path

array([[ 5.        ,  5.        ],
       [ 6.        ,  5.        ],
       [ 7.        ,  5.        ],
       [ 8.        ,  5.        ],
       [ 9.        ,  5.        ],
       [10.        ,  5.        ],
       [11.        ,  5.        ],
       [12.        ,  5.        ],
       [13.        ,  5.        ],
       [14.        ,  5.        ],
       [15.        ,  5.        ],
       [15.        ,  6.        ],
       [15.        ,  7.        ],
       [15.        ,  8.        ],
       [15.        ,  9.        ],
       [15.        , 10.        ],
       [15.        , 11.        ],
       [15.        , 12.        ],
       [15.        , 13.        ],
       [15.        , 14.        ],
       [15.        , 15.        ],
       [14.        , 15.        ],
       [13.        , 15.        ],
       [12.        , 15.        ],
       [11.        , 15.        ],
       [10.        , 15.        ],
       [ 9.        , 15.        ],
       [ 8.        , 15.        ],
       [ 7.        ,

In [508]:
#define link lengths
l1 = 25 # base link
l2 = 25 # tip link

#distance from origin to tip
r_squared = tip_path[:,0]**2 + tip_path[:,1]**2

#law of cos for angle between links
alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
print(alpha_temp)
alpha = np.arccos(alpha_temp)

print(alpha*rtd)

#vertical angle theorem for theta 2
theta2 = 180 - alpha*rtd

#triangle in link1 co-ordinant system for psi
psi = np.arctan2(l2*sind(theta2),l1+l2*cosd(theta2))

#angle of r to x-axis
beta = np.arctan2(tip_path[:,1],tip_path[:,0])*rtd

#difference in beta and psi is theta 1
theta1 = beta - psi

##theta2 = 180 - theta2

print("\ntheta 1:\n",theta1)
print("theta 2:\n",theta2)

[0.96       0.9512     0.9408     0.9288     0.9152     0.9
 0.8832     0.8648     0.8448     0.8232     0.8        0.7912
 0.7808     0.7688     0.7552     0.74       0.7232     0.7048
 0.6848     0.6632     0.64       0.6632     0.6848     0.7048
 0.7232     0.74       0.7552     0.7688     0.7808     0.7912
 0.8        0.82567901 0.84938272 0.87111111 0.8908642  0.90864198
 0.92444444 0.9382716  0.95012346 0.96      ]
[16.26020471 17.97337721 19.81365713 21.75147702 23.76585432 25.84193276
 27.96917985 30.14012429 32.34948614 34.59357902 36.86989765 37.70220482
 38.66611888 39.75374814 40.9571516  42.26858443 43.6806857  45.18661138
 46.78011969 48.45561796 50.2081805  48.45561796 46.78011969 45.18661138
 43.6806857  42.26858443 40.9571516  39.75374814 38.66611888 37.70220482
 36.86989765 34.34260675 31.85540647 29.4119844  27.01795678 24.68164795
 22.41526949 20.23670819 18.1722052  16.26020471]

theta 1:
 [43.57110073 38.39162207 34.13978824 30.62440433 27.69120398 25.21976826
 23

In [509]:
theta_min = 90
theta_max = 270
min_new = 1000
max_new = 2000

myint = min_new + ((theta1-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print(myint)


myint2 = min_new + ((theta2-theta_min)*(max_new-min_new))/(theta_max-theta_min)

#myint2_dif = 1500 - myint2
#myint2 = 1500 - myint2_dif -500
print('\n',myint2)
#print('\n',myint2_dif)

#theta2_new = theta_min + ((myint - min_new) * (theta_max - theta_min)) / (max_new - min_new)
#theta2_new

[742.06167071 713.28678928 689.66549023 670.13557959 653.84002209
 640.10982365 628.42908662 618.40050458 609.71676234 602.13840921
 595.47723917 614.2201498  632.13068163 649.15893059 665.27988211
 680.48963435 694.8012628  708.240775   720.84344694 732.65069278
 743.70751502 754.59440401 766.23918894 778.68734996 791.98084349
 806.15555073 821.23814286 837.24240908 854.16519854 871.98226662
 890.64447447 882.94457277 874.09206815 863.81822857 851.77113268
 837.48511474 820.33920705 799.50360409 773.87988454 742.06167071]

 [1409.6655294  1400.14790437 1389.92412704 1379.15846099 1367.96747599
 1356.43370687 1344.6156675  1332.55486506 1320.28063257 1307.81344986
 1295.1672353  1290.54330656 1285.18822842 1279.14584367 1272.46026887
 1265.17453095 1257.32952389 1248.9632701  1240.11044618 1230.80212245
 1221.06566389 1230.80212245 1240.11044618 1248.9632701  1257.32952389
 1265.17453095 1272.46026887 1279.14584367 1285.18822842 1290.54330656
 1295.1672353  1309.20774028 1323.02551959 

In [510]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

In [511]:
byte1 = np.zeros(len(theta1), dtype=int)
byte2 = np.zeros(len(theta1), dtype=int)
byte3 = np.zeros(len(theta1), dtype=int)
byte4 = np.zeros(len(theta1), dtype=int)

for i in range(len(theta1)):
    byte1[i], byte2[i] = break_into_two(myint[i])
    byte3[i], byte4[i] = break_into_two(myint2[i])

print(byte1,'\n\n',byte2,'\n\n\n',byte3,'\n\n',byte4)

[2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 2] 

 [230 201 177 158 141 128 116 106  97  90  83 102 120 137 153 168 182 196
 208 220 231 242 254  10  23  38  53  69  86 103 122 114 106  95  83  69
  52  31   5 230] 


 [5 5 5 5 5 5 5 5 5 5 5 5 5 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5 5
 5 5 5] 

 [129 120 109  99  87  76  64  52  40  27  15  10   5 255 248 241 233 224
 216 206 197 206 216 224 233 241 248 255   5  10  15  29  43  56  69  82
  95 107 119 129]


In [513]:
# Send all path points to both servos
for i in range(len(theta1)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)

    # read back confirmation from Arduino
    line1 = read_one_line(ser)  # servo 1 bytes echo
    line2 = read_one_line(ser)  # servo 1 int echo
    line3 = read_one_line(ser)  # servo 2 bytes echo
    line4 = read_one_line(ser)  # servo 2 int echo
    print(f"Step {i}: servo1={line2}  servo2={line4}")

    print('\ntheta1:',theta1[i],'\ntheta2:',theta2[i],'\n','\n\n')

    time.sleep(0.5)  # pause between steps so servo has time to move

Step 0: servo1=742  servo2=1409

theta1: 43.57110072780927 
theta2: 163.73979529168804 
 


Step 1: servo1=713  servo2=1400

theta1: 38.391622070508696 
theta2: 162.0266227865969 
 


Step 2: servo1=689  servo2=1389

theta1: 34.13978824209461 
theta2: 160.1863428671063 
 


Step 3: servo1=670  servo2=1379

theta1: 30.62440432689322 
theta2: 158.2485229778019 
 


Step 4: servo1=653  servo2=1367

theta1: 27.691203976009106 
theta2: 156.23414567883134 
 


Step 5: servo1=640  servo2=1356

theta1: 25.219768256181226 
theta2: 154.15806723683286 
 


Step 6: servo1=628  servo2=1344

theta1: 23.11723559236128 
theta2: 152.03082014917922 
 


Step 7: servo1=618  servo2=1332

theta1: 21.312090824153998 
theta2: 149.85987571022233 
 


Step 8: servo1=609  servo2=1320

theta1: 19.749017220844735 
theta2: 147.65051386204243 
 


Step 9: servo1=602  servo2=1307

theta1: 18.38491365827452 
theta2: 145.4064209751653 
 


Step 10: servo1=595  servo2=1295

theta1: 17.185903050523756 
theta2: 143.13010

In [66]:
#byte3, byte4 = break_into_two(1200)

In [67]:
#WriteByte(ser, MSB)
#WriteByte(ser, LSB)

In [69]:

WriteByte(ser,byte1)#<--
time.sleep(0.1)

WriteByte(ser,byte2)#<--
time.sleep(0.1)

#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

AttributeError: 'numpy.ndarray' object has no attribute 'to_bytes'

In [ ]:
byte3, byte4 = break_into_two(1500)

In [ ]:
WriteByte(ser,byte3)#<--
time.sleep(0.1)

WriteByte(ser,byte4)#<--
time.sleep(0.1)
#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

In [ ]:
ser.close()

In [ ]:
for i in range(1100, 1900, 50):
    byte3, byte4 = break_into_two(i)
    WriteByte(ser,byte3)#<--
    time.sleep(0.1)

    WriteByte(ser,byte4)#<--
    time.sleep(0.1)
    #WriteByte(ser,byte3)#<--
    #time.sleep(0.1)

    #WriteByte(ser,byte4)#<--
    #time.sleep(0.1)
    next_line = read_one_line(ser)
    extra = read_all(ser)
    print('next_line: %s' % next_line)
    print('extra: %s' % extra)
    time.sleep(0.5)
    print(i) 

- How do we break this into two bytes?
- How do we find the most significant byte?
- How do we find the least significant byte?

In [ ]:
ser.close()